# 4 — When the adjoint blows up: reading the growth rate, and turning terms off

An ocean model is differentiable, but that does not mean every gradient it gives you is
usable. Run the reverse pass far enough and it can grow until the answer is dominated by
amplified round-off rather than by physics. This notebook shows how to *see* that happen,
how to find which term is responsible, and how to switch that term's contribution off in
the backward pass without touching the forward model at all.

The short version, for the impatient: on the realistic global configuration the reverse
pass grew by about 1.3x per 30-minute timestep, which is a hundred million over two days,
and the two-day gradient flipped sign between runs that were supposed to be identical.
The cause was the turbulence closure's response to the ocean state. Freezing it in the
backward brought the growth down to 1.007x per step and the gradient repeated to six
significant figures. That fix is one flag.


## The idea

`stop_gradient` is the identity in the forward direction. If you apply it to a quantity
the model computes, the model still computes exactly the same number and the trajectory
does not move by a single bit — but the reverse pass stops propagating through whatever
produced it. So you can remove a term's *sensitivity* while keeping its *value*.

This is not a trick invented here. It is what the ECCO ocean state estimation project
does with the mixing coefficients: they use the diffusivities and viscosities the
turbulence closure computes, but omit the closure's dependence on the ocean state when
they differentiate. It is how they get a stable adjoint over a 26-year integration.

`fesom_jax` exposes this as one object with one flag per term.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import fesom_jax
from fesom_jax import forcing, ic, ssh
from fesom_jax.adjoint_stab import FREEZE_FLAGS, AdjointConfig
from fesom_jax.integrate import integrate
from fesom_jax.mesh import load_mesh
from fesom_jax.params import Params
from fesom_jax.config import A_VER

print('flags available:')
for f in FREEZE_FLAGS:
    print('   ', f)


Each flag means the same thing: *use the value this term computes, but do not let the
gradient flow through how it was computed.* All default to off, and with everything off
the traced computation is exactly what it was before this feature existed.


## A model to differentiate

The packaged `pi` mesh is a small idealized global ocean — a few thousand nodes, fast
enough to run in a notebook. The loss is the mean sea-surface temperature after a few
steps, and we differentiate it with respect to the background vertical diffusivity.


In [ ]:
mesh   = load_mesh()                     # the packaged pi mesh
DT, N  = 100.0, 8
op     = ssh.build_ssh_operator(mesh, dt=DT)
stress = forcing.surface_stress(mesh)
st0    = ic.initial_state(mesh)

def make_loss(cfg):
    def loss(k_ver):
        p = Params(k_ver=k_ver, a_ver=jnp.asarray(A_VER, jnp.float64))
        fin = integrate(st0, mesh, op, stress, n_steps=N, params=p, dt=DT,
                        adjoint_cfg=cfg)          # <- the only new argument
        return jnp.mean(fin.T[:, 0])
    return jax.jit(loss), jax.jit(jax.grad(loss))

k0 = jnp.asarray(1.0e-4, jnp.float64)
loss_exact, grad_exact = make_loss(None)
print(f'loss     = {float(loss_exact(k0)):.12e}')
print(f'd/dk_ver = {float(grad_exact(k0)):+.6e}')


## The property that makes this safe

Before trusting any of these flags, check the thing that must always hold: turning one on
changes the gradient and leaves the forward untouched. Not *nearly* untouched — bit for
bit identical. If that ever fails, the flag is wired wrong and is silently changing your
physics.


In [ ]:
rows = []
for flag in FREEZE_FLAGS:
    l, g = make_loss(AdjointConfig(**{flag: True}))
    rows.append((flag, float(l(k0)), float(g(k0))))

l0, g0 = float(loss_exact(k0)), float(grad_exact(k0))
print(f"{'flag':<16}{'forward identical?':<20}{'gradient':>14}{'change':>10}")
print(f"{'(none)':<16}{'--':<20}{g0:>14.6e}{'--':>10}")
for flag, l, g in rows:
    same = 'yes' if l == l0 else f'NO ({l-l0:+.3e})'
    rel  = '--' if g == g0 else f'{(g-g0)/abs(g0)*100:+.1f}%'
    print(f'{flag:<16}{same:<20}{g:>14.6e}{rel:>10}')


Every row should say `yes` in the second column. The flags that show no change in the
gradient are the ones whose term is not on this configuration's code path — the small
idealized setup runs the simple Pacanowski–Philander mixing rather than the prognostic
turbulence closure, and has no eddy parameterization, so those flags have nothing to act
on here. On the realistic global configuration they all do something.


## Watching the reverse pass grow

The number that decides whether a long window is usable is how fast the reverse-pass field
grows per timestep. Below one, the adjoint forgets the distant past and everything is
fine. Meaningfully above one, it compounds: 1.3 per step is a factor of a hundred million
over two days at a half-hour timestep.

You can measure it directly by differentiating windows of increasing length and watching
the size of the gradient.


In [ ]:
def growth_curve(cfg, lengths=(2, 4, 8, 16, 32)):
    out = []
    for n in lengths:
        def loss(k_ver, n=n):
            p = Params(k_ver=k_ver, a_ver=jnp.asarray(A_VER, jnp.float64))
            fin = integrate(st0, mesh, op, stress, n_steps=n, params=p, dt=DT,
                            adjoint_cfg=cfg)
            return jnp.mean(fin.T[:, 0])
        out.append(abs(float(jax.jit(jax.grad(loss))(k0))))
    return np.array(lengths), np.array(out)

n_ex, g_ex = growth_curve(None)
n_fr, g_fr = growth_curve(AdjointConfig(freeze_mixing=True))

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(n_ex, g_ex, 'o-', label='everything differentiated')
ax.semilogy(n_fr, g_fr, 's-', label='mixing coefficients frozen')
ax.set_xlabel('window length (timesteps)')
ax.set_ylabel(r'$|\partial\,\mathrm{loss}/\partial k_{ver}|$')
ax.set_title('How fast does the gradient grow with window length?')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()


On this small idealized mesh both curves are tame — that is the point of a test case. On
the realistic global configuration the first curve climbs by five orders of magnitude per
simulated day and the second one is flat.

A useful way to read a curve like this: fit the slope on the log axis and convert it to a
per-step factor. Anything comfortably below about 1.01 per step means you can afford a
window of weeks; 1.3 per step means you cannot afford a window of days.


In [ ]:
def per_step_growth(n, g):
    """Least-squares growth factor per timestep from a |gradient| vs window-length curve."""
    ok = g > 0
    slope = np.polyfit(n[ok], np.log(g[ok]), 1)[0]
    return float(np.exp(slope))

for name, (n, g) in [('exact', (n_ex, g_ex)), ('mixing frozen', (n_fr, g_fr))]:
    r = per_step_growth(n, g)
    print(f'{name:<16} {r:.4f} per step   ->  x{r**48:,.3g} over a day at dt=1800 s')


## Using it in a real run

Nothing else changes. Build the config once and pass it wherever you pass the physics
configs. It is a plain hashable object, so it can be closed over inside a jitted training
step without forcing recompilation.

```python
adj = AdjointConfig(freeze_mixing=True)          # the ocean switch, measured
ice = IceConfig(whichEVP=1, adjoint_mode='rheology')   # the ice switch, measured

final = integrate(state, mesh, op, stress, n_steps=n, params=p, dt=1800.0,
                  gm_cfg=gm_cfg, tke_cfg=tke_cfg, ale_cfg=ale_cfg,
                  ice_cfg=ice, adjoint_cfg=adj)
```

Sea ice has its own switch rather than a flag on this object, because it predates it and
because it has three settings rather than two. `'rheology'` is the one to use: it
differentiates ice thermodynamics, advection, the concentration cut-off and the ocean-to-ice
coupling, and freezes only the momentum solver, whose subcycled relaxation is what makes an
exact ice adjoint return `NaN`. `'frozen'` freezes the whole ice update and is the
conservative fallback; `'exact'` is the mathematically faithful setting and is unusable on
realistic configurations.

[`05_five_day_adjoint.ipynb`](05_five_day_adjoint.ipynb) puts both switches together on the
real global ocean, using the public data release.


## What you give up

Be clear about the trade. Freezing a term does not give you a more accurate gradient; it
gives you the exact gradient of a slightly different model, one in which that feedback
loop has been cut. Whether that is acceptable depends on what you are training.

For an eddy parameterization it is a good trade, and not by luck: the eddy coefficients
reach the vertical diffusion through a term that is added *after* the turbulence closure
runs, so freezing the closure leaves the parameter's own gradient path completely intact.
What gets cut is a separate feedback loop that was contributing amplification and very
little signal.

For tuning the turbulence closure itself it would obviously be the wrong thing to do —
you would be deleting exactly the sensitivity you are trying to learn from.

Two flags deserve a warning label. `k33` cuts part of the eddy coefficients' own gradient
path, so it is for isolating that term, not for training. `pgf` removes the coupling from
temperature and salinity into the currents entirely, which is a diagnostic only.

See `docs/ADJOINT_HORIZON.md` for the measured numbers on the realistic configuration and
for what the ocean state estimation literature does about this.
